# Quick SLM — 04b · SFT corpus fixer (salvage what is already generated)

You already spent teacher time and have raw shards on Drive (about 30k single_stage lines).
Do not regenerate them. This notebook repairs the corpus you have, on CPU, with no teacher:
it re-runs the corrected validation and the dedup pass that collapses the near-duplicates the
small seed pools produced, reports where the examples went per cell, and optionally rebalances
and packs the result into training-ready windows.

It never loads the teacher and never calls plan_requests or run_generation, so it cannot
regenerate or overwrite a raw shard. Safe to run on a CPU runtime.

Honest limit: dedup removes duplicates, it cannot create diversity. The direct subtype will
shrink the most, because the same seed, plainly phrased, produces the same example. This
salvages every distinct example you paid for; any shortfall to your target counts has to be
generated later, and the per-seed cap from notebook 04a is what makes that efficient.

## 1 · Framework (no GPU)

Same install as notebook 04. A CPU runtime is enough; nothing here touches the teacher.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXTRAS = 'sft'
DRIVE_ROOT = '/content/drive/MyDrive/quick-slm'

import subprocess, sys, importlib
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / 'code', root, root / 'quick-slm']
REPO_DIR = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if REPO_DIR is None:
    raise RuntimeError('No pyproject.toml on Drive under ' + str(root / 'code') + '; upload the repo there and re-run.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR) + '[' + EXTRAS + ']'], check=True)

framework_dir = REPO_DIR / 'framework'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))
importlib.invalidate_caches()

import v1.quick_slm_trainer as q
print('quick-slm-trainer', q.__version__, 'from', Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, 'require_framework'):
    q.require_framework('v1', REPO_DIR)
else:
    raise RuntimeError(
        'quick-slm-trainer ' + q.__version__ + ' predates the support-window check; '
        'training v1 requires >=1.0. See SUPPORT.md.'
    )


## 2 · What is on disk

Count the raw teacher lines already on Drive, the ones you do not want to recreate. Layout
defaults to the quick-slm Drive root, the same place notebook 04 copies finished
shards to. If your corpus lives elsewhere, pass `Layout(drive_root=Path(...))`.

In [ ]:
import collections
from v1.quick_slm_trainer import Layout, sft_v1
from v1.quick_slm_trainer.sft.generate import read_shard
from v1.quick_slm_trainer.sft.specs import CATEGORIES

layout = Layout().mkdirs_sft()
cfg = sft_v1()

print('reading raw shards from:', str(layout.sft_raw('single_stage').parent))
print()
raw_rows = {}
for cat in CATEGORIES:
    rows = list(read_shard(layout.sft_raw(cat)))
    raw_rows[cat] = rows
    local_n = len(list(read_shard(layout.local_sft_raw(cat))))
    note = '' if local_n == len(rows) else '   (local copy has ' + f'{local_n:,}' + ')'
    print(cat.ljust(24) + f'{len(rows):>9,} raw lines' + note)
total_raw = sum(len(v) for v in raw_rows.values())
print()
print(f'total raw lines on disk: {total_raw:,}')
print()
print('single_stage raw, per (domain, subtype):')
c = collections.Counter((row.get('domain', ''), row.get('subtype', '')) for row in raw_rows['single_stage'])
for key in sorted(c):
    d, s = key
    print('  ' + d.ljust(9) + s.ljust(22) + f'{c[key]:>9,}')

## 3 · Validate

Re-run the filters on the raw teacher text. Validation is re-runnable by design: the raw
shard stores what the teacher said, and every corrected filter (the per-spec schema, the
structural checks) applies for free on a re-read, with no regeneration. Category 3 is judged
pairwise from the oracle recomputed here, not from anything in the shard.

In [ ]:
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram, subtype_histogram, paired_integrity

examples, stats, per_category = load_and_validate(layout, cfg.sft)
print()
print('=== validation, corpus-wide ===')
print(stats.report())
print()
print('accepted by category:', category_histogram(examples))

## 4 · Dedup, the actual fix

MinHash over the user request plus the call signatures, Jaccard >= 0.85, exactly as notebook
04. This is the pass that collapses the seed-reuse near-duplicates. The two branches of a
counterfactual pair move together, so a pair is kept or dropped whole.

In [ ]:
from v1.quick_slm_trainer.sft.dedup import dedup_examples

before = len(examples)
before_sub = subtype_histogram(examples)

deduped = dedup_examples(examples, threshold=cfg.sft.dedup_jaccard, num_perm=cfg.sft.minhash_perms, progress=True)

after = len(deduped)
after_sub = subtype_histogram(deduped)
pct = 100 * (before - after) / max(before, 1)
print()
print(f'{before:,} validated -> {after:,} distinct   ({pct:.1f}% near-duplicate removed)')
print('after dedup by category:', category_histogram(deduped))

n_pairs, broken = paired_integrity(deduped)
print('counterfactual pairs intact:', n_pairs, '  broken:', len(broken))

## 5 · The funnel, per cell

raw on disk, then validated, then distinct after dedup, per (category, subtype). This is the
map of where your 30k went, and which cells collapsed.

In [ ]:
raw_sub = collections.Counter((row.get('category', ''), row.get('subtype', '')) for rows in raw_rows.values() for row in rows)

head = 'category'.ljust(16) + 'subtype'.ljust(22) + 'raw'.rjust(9) + 'validated'.rjust(11) + 'deduped'.rjust(9) + 'kept'.rjust(7)
print(head)
print('-' * len(head))
for cat in CATEGORIES:
    subs = set(before_sub.get(cat, {})) | {s for (cc, s) in raw_sub if cc == cat} | set(after_sub.get(cat, {}))
    for s in sorted(subs):
        r = raw_sub.get((cat, s), 0)
        v = before_sub.get(cat, {}).get(s, 0)
        d = after_sub.get(cat, {}).get(s, 0)
        kept = f'{100 * d / r:.0f}%' if r else '-'
        print(cat.ljust(16) + s.ljust(22) + f'{r:>9,}' + f'{v:>11,}' + f'{d:>9,}' + kept.rjust(7))
print()
print('kept = deduped / raw. The direct subtype is where seed reuse bites: same seed, same plain')
print('phrasing, same call, so dedup collapses it hardest. This salvages the distinct signal but')
print('cannot manufacture the diversity that was never generated.')

## 6 · Optional rebalance (off by default)

After dedup the subtype mix is uneven, because direct collapsed further than paraphrased. If
you want no single cell to dominate what the student sees, cap each unpaired subtype. Paired
examples are left untouched so no pair is split. Off by default: set REBALANCE = True to apply.

In [ ]:
REBALANCE = False        # trim over-represented cells so no subtype dominates the mix
PER_SUBTYPE_CAP = 1500   # applied to unpaired categories only when REBALANCE is True

import random
corpus = list(deduped)

if REBALANCE:
    rng = random.Random(20240201)
    paired_ex = [e for e in corpus if e.category == 'state_memory_conflict']
    unpaired_ex = [e for e in corpus if e.category != 'state_memory_conflict']

    buckets = collections.defaultdict(list)
    for e in unpaired_ex:
        buckets[(e.category, e.meta.get('subtype', ''))].append(e)

    kept_ex = []
    for key in sorted(buckets):
        xs = buckets[key]
        if len(xs) > PER_SUBTYPE_CAP:
            idx = list(range(len(xs)))
            rng.shuffle(idx)
            xs = [xs[i] for i in sorted(idx[:PER_SUBTYPE_CAP])]
        kept_ex.extend(xs)

    corpus = paired_ex + kept_ex
    print('after per-subtype cap of', PER_SUBTYPE_CAP, ':', category_histogram(corpus), ' total', len(corpus))
    n_pairs, broken = paired_integrity(corpus)
    print('pairs intact:', n_pairs, '  broken:', len(broken))
else:
    print('REBALANCE is False; corpus is the deduped set unchanged:', len(corpus), 'examples.')

## 7 · Optional pack (off by default, writes to Drive)

Split at the example level and pack into ctx-width windows, the same as notebook 04 section 8,
so the salvaged corpus is ready for training in notebook 05. This writes .bin files to Drive,
so it is opt-in: set PACK = True to run it.

In [ ]:
PACK = False   # writes packed train/val windows to Drive (mirrors notebook 04 section 8)

if PACK:
    from v1.quick_slm_trainer.sft.pack import pack_split, split_examples, write_stats
    from v1.quick_slm_trainer.tokenizer import load_tokenizer

    tok = load_tokenizer(layout.tokenizer_dir, patch=False)
    cfg.sft.ctx = cfg.data.ctx

    train_ex, val_ex = split_examples(corpus, val_fraction=cfg.sft.val_fraction)
    for name, split in (('train', train_ex), ('val', val_ex)):
        _, bad = paired_integrity(split)
        assert not bad, name + ' split has a lone branch: ' + str(bad[:5])
    print('train', len(train_ex), '  val', len(val_ex))

    train_stats = pack_split(layout, 'train', train_ex, tok, cfg.sft)
    print()
    val_stats = pack_split(layout, 'val', val_ex, tok, cfg.sft)

    path = write_stats(layout, {
        'source': 'salvaged from existing raw shards by 04b_sft_fix_corpus',
        'config': cfg.sft.to_dict(),
        'after_dedup': {'examples': len(deduped), 'by_category': category_histogram(deduped), 'by_subtype': subtype_histogram(deduped)},
        'after_rebalance': {'examples': len(corpus), 'by_category': category_histogram(corpus)},
        'pack': {'train': train_stats.to_dict(), 'val': val_stats.to_dict()},
    })
    print()
    print('wrote', path)
    print('packed train tokens:', f'{train_stats.total_tokens:,}')
    print('packed val tokens  :', f'{val_stats.total_tokens:,}')
else:
    print('PACK is False. Set it True to write training-ready windows to Drive for notebook 05.')